# 03 — Feature Extraction

Two feature sets are computed:
- **Option A**: CSP log-variance per band (6 bands × 6 components = 36 features)
- **Option B**: Welch PSD band power per channel (5 bands × n_channels, PCA-reduced)

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np

from features import CSPBandFeatures, band_power_features
from utils import set_seed

set_seed(42)
DATA_DIR = Path('../data')

X = np.load(DATA_DIR / 'X.npy')
y = np.load(DATA_DIR / 'y.npy')
print('X:', X.shape, 'y:', y.shape)

## Option A — CSP band-power features

In [ ]:
csp = CSPBandFeatures(n_components=6)
X_csp = csp.fit_transform(X, y)
print('CSP features shape:', X_csp.shape)
print('Feature names:', csp.feature_names()[:10], '...')

np.save(DATA_DIR / 'X_csp.npy', X_csp)

## Option B — PSD band-power features

In [ ]:
X_psd = band_power_features(X)
print('PSD features shape:', X_psd.shape)

from sklearn.decomposition import PCA
pca = PCA(n_components=50, random_state=42)
X_psd_reduced = pca.fit_transform(X_psd)
print('PCA-reduced PSD features:', X_psd_reduced.shape)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.1%}')

np.save(DATA_DIR / 'X_psd.npy', X_psd_reduced)

## Quick sanity check — class separability (PCA scatter)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(X_csp[mask, 0], X_csp[mask, 1], label=f'class {cls}', alpha=0.6)
ax.set_xlabel('alpha-CSP1')
ax.set_ylabel('alpha-CSP2')
ax.legend()
ax.set_title('CSP feature separability (alpha band)')
plt.show()